In [1]:
import time
import numpy as np
import gymnasium as gym
from stable_baselines3 import PPO
from torch.utils.tensorboard import SummaryWriter
from gymnasium.wrappers import RecordVideo
import shooter

MODEL_PATH = "GuardianPPO_sil_failure"
LOG_DIR    = "runs/eval_guardian"
VIDEO_DIR  = "videos/guardian_best"
NUM_EPS    = 10

model = PPO.load(MODEL_PATH)
writer = SummaryWriter(LOG_DIR)

# Phase 1 – evaluate with fixed seeds, store seed for each episode
results = []          # (total_reward, final_score, kills, steps, seed)
print(f"Evaluating GuardianPPO – {NUM_EPS} episodes …\n")
for ep in range(NUM_EPS):
    ep_seed = 4000 + ep          # fixed seed, different range from V5 to avoid collision
    env = gym.make("Shooter-v0", render_mode=None)
    obs, info = env.reset(seed=ep_seed)
    total_reward, ep_kills = 0.0, 0
    prev_score = info["hunterScore"]

    while True:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        score_delta = info["hunterScore"] - prev_score
        if score_delta > 0:
            ep_kills += 1
        prev_score = info["hunterScore"]
        if terminated or truncated:
            break
    env.close()

    final_score = info["hunterScore"]
    steps = info["tick"]
    results.append((total_reward, final_score, ep_kills, steps, ep_seed))

    # Log each episode to TensorBoard
    writer.add_scalar("Episode/Reward", total_reward, ep)
    writer.add_scalar("Episode/Score",  final_score,  ep)
    writer.add_scalar("Episode/Kills",  ep_kills,     ep)
    writer.add_scalar("Episode/Steps",  steps,        ep)

# Convert to numpy arrays for analysis
arr_rewards = np.array([r[0] for r in results])
arr_scores  = np.array([r[1] for r in results])
arr_kills   = np.array([r[2] for r in results])
arr_steps   = np.array([r[3] for r in results])

best_reward_idx = np.argmax(arr_rewards)
best_score_idx  = np.argmax(arr_scores)
best_kill_idx   = np.argmax(arr_kills)

# Print summary table
print("─" * 60)
print(f"{'Ep':>4s} | {'Reward':>10s} | {'Score':>6s} | {'Kills':>5s} | {'Steps':>5s} | {'Seed':>5s}")
print("─" * 60)
for i in range(NUM_EPS):
    print(f"{i+1:4d} | {arr_rewards[i]:10.2f} | {arr_scores[i]:6d} | {arr_kills[i]:5d} | {arr_steps[i]:5d} | {results[i][4]:5d}")
print("─" * 60)
print(f"Best Reward : episode {best_reward_idx+1:2d}  with reward = {arr_rewards[best_reward_idx]:.2f}  (seed {results[best_reward_idx][4]})")
print(f"Best Score  : episode {best_score_idx+1:2d}  with score  = {arr_scores[best_score_idx]}  (seed {results[best_score_idx][4]})")
print(f"Best Kills  : episode {best_kill_idx+1:2d}  with kills  = {arr_kills[best_kill_idx]}  (seed {results[best_kill_idx][4]})")
print("─" * 60)

# Log overall summary scalars
writer.add_scalar("Summary/Mean_Reward", np.mean(arr_rewards))
writer.add_scalar("Summary/Std_Reward",  np.std(arr_rewards))
writer.add_scalar("Summary/Mean_Score",  np.mean(arr_scores))
writer.add_scalar("Summary/Mean_Kills",  np.mean(arr_kills))

# Phase 2 – record the exact episode with the BEST SCORE
chosen_ep = best_score_idx               # best score, not reward
best_seed = results[chosen_ep][4]
print(f"\nRecording episode {chosen_ep+1} (best score) with seed={best_seed} (score will be {arr_scores[chosen_ep]}) …")

# Create a fresh environment with the RecordVideo wrapper
env = gym.make("Shooter-v0", render_mode="rgb_array")
env = RecordVideo(
    env, VIDEO_DIR,
    episode_trigger=lambda ep_id: ep_id == 0   # record the very first episode
)
obs, info = env.reset(seed=best_seed)          # same seed as Phase 1 ⇒ identical episode

while True:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        break
env.close()

writer.close()
print(f"\nVideo saved in '{VIDEO_DIR}'.")
print(f"Run `tensorboard --logdir runs` to view logs.")

2026-05-12 12:09:15.200418: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


ModuleNotFoundError: No module named 'shooter'